In [1]:
from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

# 사용자 동작(액션)을 구현하기 위한 클래스
from selenium.webdriver.common.action_chains import ActionChains

# 특정 엘리먼트에 마우스 휠 내리게 하는 동작
from selenium.webdriver.common.actions.wheel_input import ScrollOrigin
from tqdm import tqdm

In [3]:
def preprocess_sentence_kr(w):
    w = w.strip()
    w = re.sub(r"[^0-9가-힣?.!,]+" , " ",w)
    w = w.strip()
    return w

In [5]:
driver = wb.Chrome()

url = "https://play.google.com/store/apps/details?id=com.lgeha.nuts&hl=ko"
driver.get(url)
print("브라우저가 열렸습니다.")

브라우저가 열렸습니다.


In [6]:
driver.maximize_window()

In [15]:
# 리뷰 더보기 버튼 클릭!!

# 방법 1.
more_btn = driver.find_element(By.CSS_SELECTOR, "#yDmH0d > c-wiz.SSPGKf.Czez9d > div > div > div:nth-child(1) > div > div.wkMJlb.YWi3ub > div > div.qZmL0 > div:nth-child(1) > c-wiz:nth-child(5) > section > header > div > div:nth-child(2) > button")
more_btn.click()


In [13]:
# 리뷰 더보기 버튼 클릭!!

# 방법 2. 리스트로 받아서 찾아주기 
more_btn = driver.find_elements(By.CSS_SELECTOR,"i.VfPpkd-kBDsod.W7A5Qb")
len(more_btn)
more_btn[2].click()

ElementClickInterceptedException: Message: element click intercepted: Element <i class="google-material-icons notranslate VfPpkd-kBDsod W7A5Qb" aria-hidden="true">...</i> is not clickable at point (269, 654). Other element would receive the click: <div class="VfPpkd-IE5DDf" jsname="GGAcbc" jsaction="click:KY1IRb"></div>
  (Session info: chrome=131.0.6778.205)
Stacktrace:
	GetHandleVerifier [0x00007FF6CEDA80D5+2992373]
	(No symbol) [0x00007FF6CEA3BFD0]
	(No symbol) [0x00007FF6CE8D590A]
	(No symbol) [0x00007FF6CE930F2E]
	(No symbol) [0x00007FF6CE92E9CC]
	(No symbol) [0x00007FF6CE92BBA6]
	(No symbol) [0x00007FF6CE92AB01]
	(No symbol) [0x00007FF6CE91CD40]
	(No symbol) [0x00007FF6CE94F36A]
	(No symbol) [0x00007FF6CE91C596]
	(No symbol) [0x00007FF6CE94F580]
	(No symbol) [0x00007FF6CE96F584]
	(No symbol) [0x00007FF6CE94F113]
	(No symbol) [0x00007FF6CE91A918]
	(No symbol) [0x00007FF6CE91BA81]
	GetHandleVerifier [0x00007FF6CEE06A2D+3379789]
	GetHandleVerifier [0x00007FF6CEE1C32D+3468109]
	GetHa

In [17]:
# 관련성 

sort_btn = driver.find_element(By.CSS_SELECTOR, '#sortBy_1')
sort_btn.click()

In [29]:
# 클래스가 .jO7h3c인 것 중 최신순은 2번째이기에 [1]로 접근
latest_btn = driver.find_element(By.CSS_SELECTOR, '#yDmH0d > div.VfPpkd-Sx9Kwc.cC1eCc.UDxLd.PzCPDd.HQdjr.VfPpkd-Sx9Kwc-OWXEXe-FNFY6c > div.VfPpkd-wzTsW > div > div > div > div > div.fysCi.Vk3ZVd > div.JPdR6b.e5Emjc.ah7Sve.qjTEB > div > div > span:nth-child(2) > div.uyYuVb.oJeWuf > div.jO7h3c')
latest_btn.click()

# 스크롤 해보기

## driver.execute_script 
- selenium을 사용하여 자바스크립트를 실행할 수 있는 메서드

## document.querySelector() 
- html에서 클래스를 가진 모달 요소를 전달
- html내에서 첫번째 일치하는 요소를 반환

## scrollHeight
- 선택된 요소의 전체 높이를 반환
- 화면에 보이지 않는 스크롤 가능한 영역의 높이까지 포함 

In [44]:
#1. 모달 높이 가져오기

# 여러 줄의 문자열을 사용하고 싶을때 
old_height = driver.execute_script("""
    const modal = document.querySelector('div.odk6He')
    return modal.scrollHeight

""")

print ("모달높이 :", old_height )

모달높이 : 11155


# 2. 스크롤 해보기

*ActionChains(driver)*
- 인스턴스 생성
- driver 객체를 기반으로 동작을 수행할 준비
- actions 객체에 마우스와 키보드 동작을 체인으로 연결하여 실행

*scroll_from_origin*
- 특정 요소를 기준으로 스크롤 동작을 수행
- 파라미터
  * scroll_origin: 스크롤 시작 기준이 되는 요소
  * x_offset : 가로로 스크롤 할 픽셀 값
  * y_offset : 세로로 스크롤 할 픽셀 값

*perform()*
- 체인으로 연결된 동작을 실행
- actions 객체에 추가된 모든 동작을 한번에 실행

In [47]:
height =0
actions = ActionChains(driver)

In [51]:
import time
# 모달 요소 선택 -> 이게 스크롤 할 수 있는 영역을 선택하는 거야!
modal = driver.find_element(By.CSS_SELECTOR,'div.odk6He')

# scrollOrigin 생성 -> 그 스크롤 할 수 있는 영역 (모달영역)을 스크롤 영역으로 선택!
scroll_origin = ScrollOrigin.from_element(modal)
actions = ActionChains(driver) # actions 객체에 마우스와 키보드 동작을 체인으로 연결하여 실행

# 스크롤 실행 
actions.scroll_from_origin(scroll_origin,0,500).perform()
time.sleep(2)

print("scroll 완료")

scroll 완료


# 3단계

1. 초기 높이 가져오기
   * 모달의 초기 높이를 scrollHeight로 가져오기
2. while 비교문
3. 높이 비교
   * 최대 5번 스크롤 수행 , 높이에 변화가 없으면 종료
   * time.sleep(2)
4. 횟수 

In [ ]:
old_height = driver.execute_script("""
    const modal = document.querySelector('div.odk6He')
    return modal.scrollHeight

""")

count = 0 #스크롤 횟수
total_scrolls = 5

while count <= total_scrolls :
    # 스크롤 실행
    height = height + 20000
    actions.scroll_from_origin(scroll_origin,0,height).perform()
    # time
    time.sleep(2)
    

    # 새로운 높이 가져오기
    new_height = driver.execute_script("""
    const modal = document.querySelector('div.odk6He')
    return modal.scrollHeight
""")
    # 높이에 변화 없으면 종료 
    if (new_height == old_height):
        old_height = new_height
        break
    # 높이 업데이트
    else:
        count = count +1
    # 변화 있으면 count 횟수 증가

### 통합

*tqdm*
- total = total_scrolls
  * 진행 상태 표시바의 전체 작업력을 설정
  * total = 100 ->
- desc ='진행상항'
- 

In [74]:
old_height = driver.execute_script("""
    const modal = document.querySelector('div.odk6He')
    return modal.scrollHeight

""")

count = 0 #스크롤 횟수
total_scrolls = 100
bar = tqdm(total = total_scrolls , desc = '진행상황', leave =False)

while count <= total_scrolls :
    # 스크롤 실행
    height = height + 20000
    actions.scroll_from_origin(scroll_origin,0,height).perform()
    # time
    time.sleep(5)
    

    # 새로운 높이 가져오기
    new_height = driver.execute_script("""
    const modal = document.querySelector('div.odk6He')
    return modal.scrollHeight
""")
    # 높이에 변화 없으면 종료 
    if (new_height == old_height):
        break
    old_height = new_height
    count = count +1
    bar.update(1)

bar.close()
    # 변화 있으면 count 횟수 증가

진행상황:  11%|███████▊                                                               | 11/100 [05:13<32:05, 21.63s/it]    

ReadTimeoutError: HTTPConnectionPool(host='localhost', port=62955): Read timed out. (read timeout=120)

In [76]:
# 유저
user = driver.find_elements(By.CSS_SELECTOR,'.X5PpBb')

# 날짜
date = driver.find_elements(By.CSS_SELECTOR,'.bp9Aid')

# 리뷰
review = driver.find_elements(By.CSS_SELECTOR,'.h3YV2d')

In [78]:
len(user), len(date), len(review)

(623, 623, 623)

In [114]:
# 별점 담기

star_div = driver.find_elements(By.CSS_SELECTOR,'.iXRFPc')
star =[]

for i in star_div :
    star.append(i.get_attribute('aria-label'))

In [115]:
star

['별표 5개 만점에 4개를 받았습니다.',
 '별표 5개 만점에 1개를 받았습니다.',
 '별표 5개 만점에 1개를 받았습니다.',
 '별표 5개 만점에 1개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 1개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 4개를 받았습니다.',
 '별표 5개 만점에 1개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 4개를 받았습니다.',
 '별표 5개 만점에 1개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 2개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 3개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 4개를 받았습니다.',
 '별표 5개 만점에 1개를 받았습니다.',
 '별표 5개 만점에 1개를 받았습니다.',
 '별표 5개 만점에 1개를 받았습니다.',
 '별표 5개 만점에 1개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 1개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 1개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 3개를 받았습니다.',
 '별표 5개 만점에 5개를 받았습니다.',
 '별표 5개 만점에 1개를 받았습니다.',


In [104]:
star[0][:2] + star[0][10:12]

'별표4개'

In [118]:
for i in range(0,len(star)):
    star[i] = (star[i][:2] + star[i][10:12])

In [120]:
star

['별표4개',
 '별표1개',
 '별표1개',
 '별표1개',
 '별표5개',
 '별표5개',
 '별표1개',
 '별표5개',
 '별표4개',
 '별표1개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표4개',
 '별표1개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표2개',
 '별표5개',
 '별표3개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표4개',
 '별표1개',
 '별표1개',
 '별표1개',
 '별표1개',
 '별표5개',
 '별표1개',
 '별표5개',
 '별표1개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표3개',
 '별표5개',
 '별표1개',
 '별표5개',
 '별표5개',
 '별표1개',
 '별표5개',
 '별표5개',
 '별표3개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표4개',
 '별표5개',
 '별표5개',
 '별표4개',
 '별표3개',
 '별표5개',
 '별표4개',
 '별표5개',
 '별표4개',
 '별표1개',
 '별표4개',
 '별표5개',
 '별표1개',
 '별표5개',
 '별표1개',
 '별표1개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표1개',
 '별표1개',
 '별표5개',
 '별표1개',
 '별표1개',
 '별표1개',
 '별표1개',
 '별표2개',
 '별표1개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표5개',
 '별표4개',
 '별표1개',
 '별표3개',
 '별표5개',
 '별표1개',
 '별표1개',
 '별표5개',
 '별표5개',
 '별표1개',
 '별표1개',
 '별표1개',
 '별표5개',
 '별표5개',
 '별표4개',
 '별표5개',
 '별표5개',
 '별표1개',
 '별표5개',
 '별표1개',
 '별표1개',
 '별표1개',
 '별표5개',
 '별표5개',
 '별표1개',
 

In [144]:
date[3].text.replace('년 ',"-").replace('월 ',"-").replace('일',"")

'2025-1-8'

In [148]:
user[3].text

'sancuary kim'

In [150]:
thinkq_app_review =[]

for i in tqdm(range(3,len(user))):
    start = star[i].rfind('개')-1
    end = star[i].rfind('개')+1
    star_count = star[i][start:end]

    #날짜 형식 표준화
    # 년,월,일 -> "-"
    convert_date = date[i].text.replace('년 ',"-").replace('월 ',"-").replace('일',"")
    # 불필요한 기호 제거
    convert_review = preprocess_sentence_kr(review[i].text)
    # 리뷰 박스에 다 넣어줄게요 , user, star, convert_date, convert_review
    review_box = [user[i].text, star_count,convert_date,convert_review ]

    thinkq_app_review.append(review_box)
    


100%|████████████████████████████████████████████████████████████████████████████████| 620/620 [00:46<00:00, 13.28it/s]


In [152]:
thinkq_app_review

[['sancuary kim',
  '1개',
  '2025-1-8',
  '업데이트될때마다 와이파이연동이 풀려서 또 연결해야되고. 또연결해야됩니다 이걸 기기마다 업데이트될때마다해야하는데 앱만든사람은 생각이없습니까?'],
 ['Solo Joy', '5개', '2025-1-8', '자체 진단 정말 편리하고 좋은 기능 입니다 굿'],
 ['이명주', '5개', '2025-1-7', '편히한것같아요.'],
 ['김용훈', '1개', '2025-1-6', '가습기와 공기청정기 같이 사용이 안될까요?'],
 ['김태영', '5개', '2025-1-4', '잘 쓰고 있는데요. 위젯에 설정해두면 왜 자꾸 없어지죠?'],
 ['쑥',
  '4개',
  '2025-1-3',
  '멀리 있어도 세탁기 시간을 알수 있어 좋아요. 근데 모든 기능이 앱에서 활성화 되어 있지 않아 불편하네요'],
 ['상e',
  '1개',
  '2025-1-3',
  '진짜 어플 상쓰레기 같아요 허구한날 약관 동의하라질 않나 티비 제어도 뭐 하나 제대로 안되고 어쩜 굴지 대기업 대표 어플이라는게 이따윈지 기가막힙니다.'],
 ['이제후',
  '5개',
  '2025-1-3',
  '우리집에서 리모컨을 잠깐 잃어버렸을때 이걸로 잠깐 리모컨 기능을 쓰니까 좋았습니다 잠시 정리하다가 지우고 다시 깔았어요'],
 ['이재화', '5개', '2025-1-3', '원조경'],
 ['Jongho Lee', '5개', '2025-1-1', '좋은데요 많은 기능 업데이트 부탁드립니다'],
 ['정영화', '5개', '2025-1-1', '간단해서 조작이 쉬워요'],
 ['ES C',
  '4개',
  '2024-12-31',
  '워시타워 어플 기능 중 세탁 통살균 코치 가 제대로 작동이 안되요. 세탁횟수도 측정 못하고 통살균버튼이 죽어 있어서 통살균 버튼을 누를 수가 없네요. 에러 잡아서 업데이트 해주세요'],
 ['Dooyoung Jung',
  '1개',
  '2024-12-29',
  '스마트 루틴 기능이

In [156]:
import pandas as pd 
thinq_df = pd.DataFrame(thinkq_app_review, columns =['user', '별점', '날짜', '리뷰'])

In [158]:
thinq_df

,user,별점,날짜,리뷰
0,sancuary kim,1개,2025-1-8,업데이트될때마다 와이파이연동이 풀려서 또 연결해야되고. 또연결해야됩니다 이걸 기기마...
1,Solo Joy,5개,2025-1-8,자체 진단 정말 편리하고 좋은 기능 입니다 굿
2,이명주,5개,2025-1-7,편히한것같아요.
3,김용훈,1개,2025-1-6,가습기와 공기청정기 같이 사용이 안될까요?
4,김태영,5개,2025-1-4,잘 쓰고 있는데요. 위젯에 설정해두면 왜 자꾸 없어지죠?
...,...,...,...,...
615,아이러뷰,5개,2024-6-20,굿
616,김기윤,4개,2024-6-20,좋습니다
617,고천마,2개,2024-6-20,에어컨 켜짐 꺼짐 같은 예약 시간 입력할때 분단위 숫자 스크롤 걸림없이 더 휙휙 돌...
618,Min Seok Jang,5개,2024-6-19,좋습니다 편리하구요


In [160]:
thinq_df.to_csv("./thinq리뷰")